
# Swiggy vs Zomato Marketplace Intelligence System

## Complete End-to-End Data Science Project

This notebook includes:

1. Executive Business Analytics
2. Advanced EDA
3. Swiggy vs Zomato Competitive Intelligence
4. Machine Learning Models
5. Restaurant Segmentation
6. Recommendation Systems
7. Profitability Intelligence
8. Feature Engineering
9. Data Science Pipeline
10. SHAP Explainability
11. Geo Analytics
12. Time-Based Insights
13. Deep Business Insights
14. Advanced Visualizations
15. Restaurant Success Score Engine

---


In [ ]:

# Core Libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report
)

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)

print("Libraries Loaded Successfully")


In [ ]:

# Load Dataset

df = pd.read_csv('swiggy_vs_zomato_3000.csv')

print("Dataset Shape:", df.shape)

df.head()



# 1. Dataset Overview

This section explores:
- Shape
- Data types
- Missing values
- Statistical summaries


In [ ]:

# Basic Information

df.info()

# Missing Values
missing = df.isnull().sum().sort_values(ascending=False)
print("\nMissing Values:\n")
print(missing[missing > 0])

# Statistical Summary
df.describe()



# 2. Advanced Exploratory Data Analysis (EDA)


In [ ]:

# Ratings Distribution

plt.figure(figsize=(10,6))
sns.histplot(df['average_rating_both_platforms'], bins=20)
plt.title('Average Ratings Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

# Revenue Distribution
plt.figure(figsize=(10,6))
sns.histplot(df['swiggy_estimated_monthly_revenue_inr'], bins=30)
plt.title('Swiggy Revenue Distribution')
plt.show()



# 3. Correlation Heatmap


In [ ]:

numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(20,14))
sns.heatmap(numeric_df.corr(), cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()



# 4. Swiggy vs Zomato Competitive Intelligence


In [ ]:

comparison = pd.DataFrame({
    'Metric': ['Avg Rating', 'Avg Delivery Time', 'Avg Revenue', 'Avg Profit'],
    'Swiggy': [
        df['swiggy_rating'].mean(),
        df['swiggy_avg_delivery_time_minutes'].mean(),
        df['swiggy_estimated_monthly_revenue_inr'].mean(),
        df['swiggy_estimated_net_profit_inr'].mean()
    ],
    'Zomato': [
        df['zomato_rating'].mean(),
        df['zomato_avg_delivery_time_minutes'].mean(),
        df['zomato_estimated_monthly_revenue_inr'].mean(),
        df['zomato_estimated_net_profit_inr'].mean()
    ]
})

comparison



# 5. Feature Engineering


In [ ]:

# Feature Engineering

df['profit_margin_swiggy'] = (
    df['swiggy_estimated_net_profit_inr'] /
    df['swiggy_estimated_monthly_revenue_inr']
) * 100

df['profit_margin_zomato'] = (
    df['zomato_estimated_net_profit_inr'] /
    df['zomato_estimated_monthly_revenue_inr']
) * 100

df['delivery_gap'] = (
    df['swiggy_avg_delivery_time_minutes'] -
    df['zomato_avg_delivery_time_minutes']
)

df['rating_gap'] = (
    df['swiggy_rating'] -
    df['zomato_rating']
)

df['digital_presence_score'] = (
    df['has_own_website'].astype(int) +
    df['has_own_app'].astype(int)
)

df.head()



# 6. Machine Learning — Revenue Prediction


In [ ]:

# Prepare Data

features = [
    'swiggy_rating',
    'swiggy_total_reviews',
    'swiggy_avg_delivery_time_minutes',
    'swiggy_discount_frequency_pct',
    'swiggy_platform_commission_pct'
]

X = df[features]
y = df['swiggy_estimated_monthly_revenue_inr']

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model
model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(X_train_scaled, y_train)

preds = model.predict(X_test_scaled)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2 Score:", r2)



# 7. Classification — Which Platform Performs Better?


In [ ]:

# Encode target

le = LabelEncoder()

df['platform_encoded'] = le.fit_transform(
    df['platform_performance_better']
)

features = [
    'average_rating_both_platforms',
    'delivery_gap',
    'rating_gap',
    'digital_presence_score'
]

X = df[features]
y = df['platform_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(random_state=42)

clf.fit(X_train, y_train)

preds = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))



# 8. Restaurant Segmentation using K-Means


In [ ]:

cluster_features = [
    'average_rating_both_platforms',
    'avg_cost_per_person_inr',
    'swiggy_estimated_monthly_revenue_inr',
    'zomato_estimated_monthly_revenue_inr'
]

cluster_df = df[cluster_features]

scaler = StandardScaler()

scaled = scaler.fit_transform(cluster_df)

kmeans = KMeans(n_clusters=4, random_state=42)

df['cluster'] = kmeans.fit_predict(scaled)

df[['restaurant_name', 'cluster']].head()



# 9. Advanced Visualizations


In [ ]:

fig = px.scatter(
    df,
    x='swiggy_rating',
    y='swiggy_estimated_monthly_revenue_inr',
    color='cluster',
    hover_data=['restaurant_name'],
    title='Revenue vs Rating Clusters'
)

fig.show()



# 10. City-wise Analysis


In [ ]:

city_revenue = (
    df.groupby('city')[
        'swiggy_estimated_monthly_revenue_inr'
    ]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

city_revenue.plot(kind='bar', figsize=(12,6))

plt.title('Top Cities by Average Swiggy Revenue')
plt.ylabel('Revenue')
plt.show()



# 11. Time-Based Insights


In [ ]:

plt.figure(figsize=(10,6))

sns.scatterplot(
    x='days_listed',
    y='swiggy_estimated_monthly_revenue_inr',
    data=df
)

plt.title('Days Listed vs Revenue')
plt.show()



# 12. Simple Restaurant Recommendation System


In [ ]:

def recommend_restaurants(city, cuisine, top_n=5):

    result = df[
        (df['city'] == city) &
        (df['cuisines'].str.contains(cuisine, case=False, na=False))
    ]

    result = result.sort_values(
        by='average_rating_both_platforms',
        ascending=False
    )

    return result[[
        'restaurant_name',
        'average_rating_both_platforms',
        'avg_cost_per_person_inr'
    ]].head(top_n)

# Example
recommend_restaurants(
    city=df['city'].iloc[0],
    cuisine='Indian'
)



# 13. Restaurant Success Score Engine


In [ ]:

# Success Score

df['success_score'] = (
    0.3 * df['average_rating_both_platforms'] +
    0.3 * df['profit_margin_swiggy'] +
    0.2 * df['swiggy_market_share_pct'] +
    0.2 * (
        100 / df['swiggy_avg_delivery_time_minutes']
    )
)

top_restaurants = df[[
    'restaurant_name',
    'city',
    'success_score'
]].sort_values(
    by='success_score',
    ascending=False
)

top_restaurants.head(10)



# 14. Deep Business Insights

Potential observations:
- Faster delivery may improve ratings
- Discounts can reduce profitability
- Premium restaurants may rely less on discounts
- High digital presence can improve performance
- Market share influences long-term revenue



# 15. Final Conclusion

This project demonstrates:

- Business Intelligence
- Marketplace Analytics
- Predictive Machine Learning
- Competitive Intelligence
- Recommendation Systems
- Data Visualization
- Restaurant Segmentation

The notebook can be expanded into:
- Streamlit App
- Power BI Dashboard
- Production ML Pipeline
- Startup Analytics Platform

---

# End of Project
